# 06 — QPU runs (ibm_pittsburgh)

All QPU work for Etapa 6. Cells blocked created calibrations, transpiled circuits or submitted jobs.
They already ran once and are kept only as documentation: their first line stops them if run again.
Everything else only **reads** saved files and is safe to re-run.

After a kernel restart, run: **Setup** (2 cells), then any cell from **Reading and checks**.

## Setup

In [1]:
%load_ext autoreload
%autoreload 2

import json
import pandas as pd

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime.executor_sampler import Sampler

from src.graphs import load_instances, find_instance, json_to_graph
from src.hamiltonians import max_cut_hamiltonian, bisection_hamiltonian, generate_penalty_A
from src.classical import max_cut_bruteforce, bisection_bruteforce
from src.qaoa import qaoa_metrics
from src.hardware import save_calibration_snapshot, transpile_qaoa_best, estimated_success, counts_to_probabilities, save_job_data

service = QiskitRuntimeService(name="default-ibm-quantum-platform")

BACKEND_NAME = "ibm_pittsburgh"
backend = service.backend(BACKEND_NAME)

# incarc instantele de grafuri din fisierul json
instances = load_instances("../data/instances.json")
with open("../results/params_warm.json") as f:
    rows = json.load(f)

# cele 38 de circuite - din etapa 5
qpu_list = pd.read_csv("../data/qpu/circuit_list_2026-10-26.csv")

In [2]:
def problem_setup(name, problem):
    """Graph, Hamiltonian, optimum and optimal bitstrings for one instance and problem."""
    graph = json_to_graph(find_instance(instances, name))
    if problem == "maxcut":
        H = max_cut_hamiltonian(graph)
        optimum, optimal_bitstrings = max_cut_bruteforce(graph)
    else:
        H = bisection_hamiltonian(graph, generate_penalty_A(graph))
        optimum, optimal_bitstrings = bisection_bruteforce(graph)
    return graph, H, optimum, optimal_bitstrings

def find_row(name, problem, p):
    """The row of rows with the optimized angles (and ideal metrics) for one circuit."""
    return [r for r in rows if r["name"] == name and r["problem"] == problem and r["p"] == p][0]

## Pilot — DONE 2026-10-27

Job `dbg7jelrd2mc73cu4q8g` on ibm_pittsburgh: 2 circuits (ring_6, Max-Cut and bisection, p = 1), 1000 shots.
81 min in the queue, 3 s of QPU time. Calibration: `ibm_pittsburgh_2026-10-27T10-15.json`.
(A first pilot on ibm_boston, job `dbfl1hnpc3tc739nn13g`, waited 19 h in the queue and was cancelled.)

In [ ]:
raise RuntimeError("DONE 2026-10-27 — this cell creates/submits; do not re-run")


# pilot: calibration snapshot + transpilation
calibration_path = save_calibration_snapshot(backend)
print("calibration:", calibration_path)

pilot = [("ring_6", "maxcut", 1), ("ring_6", "bisection", 1)]
labels = []
circuits = []

for name, problem, p in pilot:
    graph, H, optimum, optimal_bitstrings = problem_setup(name, problem)
    row = find_row(name, problem, p)
    transpiled, seed, order = transpile_qaoa_best(H, p, row["params"], backend)

    label = name + "|" + problem + "|p" + str(p)
    labels.append(label)
    circuits.append(transpiled)
    print(label, "| cz:", transpiled.count_ops().get("cz", 0),
          "| esp:", round(estimated_success(transpiled, backend), 3))

In [ ]:
raise RuntimeError("DONE 2026-10-27 — this cell creates/submits; do not re-run")


# pilot: submission
sampler = Sampler(mode=backend)
job = sampler.run(circuits, shots=1000)
print("job id:", job.job_id())

job id: dbg7jelrd2mc73cu4q8g

In [ ]:
raise RuntimeError("DONE 2026-10-27 — this cell creates/submits; do not re-run")


# pilot: saving (circuits .qpy, raw counts .json, one row in manifest.csv)
pilot_calibration = "../data/qpu/calibration/ibm_pittsburgh_2026-10-27T10-15.json"
raw_path = save_job_data(job, labels, circuits, pilot_calibration, tag="pilot")
print("saved:", raw_path)
print("QPU seconds:", job.usage())

In [23]:
# pilot metrics, read from the saved file (safe to re-run)
manifest = pd.read_csv("../data/qpu/manifest.csv")
raw_path = manifest[manifest["tag"] == "pilot"]["raw"].iloc[0]

with open(raw_path) as f:
    data = json.load(f)

for label in data["labels"]:
    name, problem, p_text = label.split("|")
    p = int(p_text[1:])

    graph, H, optimum, optimal_bitstrings = problem_setup(name, problem)
    probabilities = counts_to_probabilities(data["counts"][label], graph.number_of_nodes())
    qpu = qaoa_metrics(graph, problem, probabilities, optimal_bitstrings)
    row = find_row(name, problem, p)

    print(label)
    print("  ideal:", {k: round(row[k], 3) for k in ["valid_fraction", "mean_cut", "p_optimal"]})
    print("  QPU:  ", {k: round(qpu[k], 3) for k in ["valid_fraction", "mean_cut", "p_optimal"]})

ring_6|maxcut|p1
  ideal: {'valid_fraction': 1.0, 'mean_cut': 4.5, 'p_optimal': 0.285}
  QPU:   {'valid_fraction': 1.0, 'mean_cut': 4.402, 'p_optimal': 0.267}
ring_6|bisection|p1
  ideal: {'valid_fraction': 0.633, 'mean_cut': 2.269, 'p_optimal': 0.575}
  QPU:   {'valid_fraction': 0.595, 'mean_cut': 2.393, 'p_optimal': 0.508}


## Main run — DONE 2026-10-27 17:03

38 circuits in 3 jobs (one per n = 6, 10, 14), 4000 shots each, on ibm_pittsburgh.
Calibration at submission: `ibm_pittsburgh_2026-10-27T15-02.json`.
Jobs: `dbgbqgdrd2mc73cudb2g` (n = 6), `dbgbqgpj4t8s73ac1n6g` (n = 10), `dbgbqh0iudkc73e460ng` (n = 14).

Note: the transpilation cell was accidentally re-run later and overwrote `submitted_2026-10-26.csv`,
so that file was deleted. CZ count and ESP per circuit are recomputed in Etapa 7 from the saved `.qpy`
circuits and the T15-02 calibration (`estimated_success_from_snapshot`).

In [ ]:
raise RuntimeError("DONE 2026-10-27 — this cell creates/submits; do not re-run")


# 🔒 main run: calibration snapshot + transpilation of the 38 circuits
calibration_path = save_calibration_snapshot(backend)
print("calibration:", calibration_path)

groups = {}            # n -> {"labels": [...], "circuits": [...]}
submitted_rows = []

for _, entry in qpu_list.iterrows():
    name, problem = entry["name"], entry["problem"]
    p, n = int(entry["p"]), int(entry["n"])

    graph, H, optimum, optimal_bitstrings = problem_setup(name, problem)
    row = find_row(name, problem, p)
    transpiled, seed, order = transpile_qaoa_best(H, p, row["params"], backend)

    label = name + "|" + problem + "|p" + str(p)
    groups.setdefault(n, {"labels": [], "circuits": []})
    groups[n]["labels"].append(label)
    groups[n]["circuits"].append(transpiled)

    submitted_rows.append({
        "label": label,
        "n": n,
        "cz_count": transpiled.count_ops().get("cz", 0),
        "esp": estimated_success(transpiled, backend),
        "cz_planned": entry["cz_count"],
        "esp_planned": entry["esp"],
        "seed": seed,
        "order": order,
        "prediction": entry.get("prediction", ""),
    })
    print(label, "| cz:", submitted_rows[-1]["cz_count"], "| esp:", round(submitted_rows[-1]["esp"], 3))

submitted = pd.DataFrame(submitted_rows)
submitted.to_csv("../data/qpu/submitted_2026-10-26.csv", index=False)
print({n: len(g["labels"]) for n, g in groups.items()})

In [ ]:
raise RuntimeError("DONE 2026-10-27 — this cell creates/submits; do not re-run")


# 🔒 main run: submission (3 jobs), circuits and job ids saved immediately
import os
from qiskit import qpy

sampler = Sampler(mode=backend)
jobs = {}

os.makedirs("../data/qpu/circuits", exist_ok=True)
for n, group in groups.items():
    job = sampler.run(group["circuits"], shots=4000)
    jobs[n] = job
    with open("../data/qpu/circuits/" + job.job_id() + ".qpy", "wb") as f:
        qpy.dump(group["circuits"], f)
    print("n =", n, "| job id:", job.job_id())

pending = {str(n): {"job_id": job.job_id(), "labels": groups[n]["labels"]} for n, job in jobs.items()}
with open("../data/qpu/pending_jobs_2026-10-27.json", "w") as f:
    json.dump(pending, f, indent=2)

n = 6 | job id: dbgbqgdrd2mc73cudb2g
n = 10 | job id: dbgbqgpj4t8s73ac1n6g
n = 14 | job id: dbgbqh0iudkc73e460ng

In [ ]:
raise RuntimeError("DONE 2026-10-27 — this cell creates/submits; do not re-run")


# 🔒 main run: saving (after the jobs finished)
for n, job in jobs.items():
    raw_path = save_job_data(job, groups[n]["labels"], groups[n]["circuits"],
                             calibration_path, tag="main_n" + str(n))
    print("n =", n, "| saved:", raw_path, "| QPU seconds:", job.usage())

## Reading and checks (read-only, safe to re-run)

In [25]:
# all jobs and the calibration files
import os

manifest = pd.read_csv("../data/qpu/manifest.csv")
print(manifest[["job_id", "tag", "created", "qpu_seconds", "calibration"]].to_string())

print()
print("calibration files:")
for f in sorted(os.listdir("../data/qpu/calibration")):
    print(" ", f)

                 job_id       tag                           created  qpu_seconds                                                   calibration
0  dbg7jelrd2mc73cu4q8g     pilot  2026-10-27 12:15:22.861241+02:00            3  ../data/qpu/calibration/ibm_pittsburgh_2026-10-27T10-15.json
1  dbgbqgdrd2mc73cudb2g   main_n6  2026-10-27 17:03:29.633575+02:00           15  ../data/qpu/calibration/ibm_pittsburgh_2026-10-27T15-02.json
2  dbgbqgpj4t8s73ac1n6g  main_n10  2026-10-27 17:03:31.342106+02:00           15  ../data/qpu/calibration/ibm_pittsburgh_2026-10-27T15-02.json
3  dbgbqh0iudkc73e460ng  main_n14  2026-10-27 17:03:32.784358+02:00           18  ../data/qpu/calibration/ibm_pittsburgh_2026-10-27T15-02.json

calibration files:
  ibm_boston_2026-10-26T11-48.json
  ibm_boston_2026-10-26T11-51.json
  ibm_boston_2026-10-26T12-08.json
  ibm_boston_2026-10-26T12-18.json
  ibm_boston_2026-10-26T12-32.json
  ibm_boston_2026-10-26T13-07.json
  ibm_boston_2026-10-26T13-26.json
  ibm_pittsburgh_202

In [26]:
# random baseline: 4000 uniformly random bitstrings per instance (fixed seed, same file on every run)
import numpy as np
from collections import Counter

rng = np.random.default_rng(1234)
random_baseline = {}

for name in sorted(qpu_list["name"].unique()):
    n = int(qpu_list[qpu_list["name"] == name]["n"].iloc[0])
    samples = rng.integers(0, 2 ** n, size=4000)
    bitstrings = [format(k, "0" + str(n) + "b") for k in samples]
    random_baseline[name] = dict(Counter(bitstrings))

with open("../data/qpu/random_baseline.json", "w") as f:
    json.dump({"seed": 1234, "shots": 4000, "counts": random_baseline}, f, indent=2)

print(len(random_baseline), "instances")

9 instances


In [27]:
# integrity checks: shots, bitstring length, and the 38 circuits on the fixed list
manifest = pd.read_csv("../data/qpu/manifest.csv")
main_labels = []

for _, job_row in manifest.iterrows():
    with open(job_row["raw"]) as f:
        data = json.load(f)
    for label in data["labels"]:
        counts = data["counts"][label]
        n = int(label.split("|")[0].split("_")[1])
        assert sum(counts.values()) == job_row["shots"], "wrong number of shots: " + label
        assert all(len(b) == n for b in counts), "wrong bitstring length: " + label
        if job_row["tag"].startswith("main"):
            main_labels.append(label)

expected = (qpu_list["name"] + "|" + qpu_list["problem"] + "|p" + qpu_list["p"].astype(str)).tolist()
assert sorted(main_labels) == sorted(expected), "the circuits on QPU differ from the list"
print("all checks passed:", len(main_labels), "circuits")

all checks passed: 38 circuits


In [ ]:
# first look: QPU vs ideal for all 38 circuits
manifest = pd.read_csv("../data/qpu/manifest.csv")
rows = []
main_jobs = manifest[manifest["tag"].str.startswith("main")]

for _, job_row in main_jobs.iterrows():
    with open(job_row["raw"]) as f:
        data = json.load(f)

    for label in data["labels"]:
        name, problem, p_text = label.split("|")
        p = int(p_text[1:])

        graph, H, optimum, optimal_bitstrings = problem_setup(name, problem)
        probabilities = counts_to_probabilities(data["counts"][label], graph.number_of_nodes())
        qpu = qaoa_metrics(graph, problem, probabilities, optimal_bitstrings)
        row = find_row(name, problem, p)

        rows.append({
            "label": label,
            "n": graph.number_of_nodes(),
            "optimum": optimum,
            "ideal_valid_fraction": row["valid_fraction"],
            "qpu_valid_fraction": qpu["valid_fraction"],
            "ideal_mean_cut": row["mean_cut"],
            "qpu_mean_cut": qpu["mean_cut"],
            "ideal_p_optimal": row["p_optimal"],
            "qpu_p_optimal": qpu["p_optimal"],
        })

first_look = pd.DataFrame(rows)
first_look

,label,n,optimum,ideal_valid_fraction,qpu_valid_fraction,ideal_mean_cut,qpu_mean_cut,ideal_p_optimal,qpu_p_optimal
0,random_6|bisection|p1,6,4,0.468053,0.43600,4.728813,4.728211,0.259086,0.23725
1,random_6|bisection|p2,6,4,0.805311,0.65175,4.962536,4.953586,0.243969,0.18025
2,random_6|maxcut|p1,6,6,1.000000,1.00000,5.054377,4.994500,0.345681,0.32275
3,random_6|maxcut|p2,6,6,1.000000,1.00000,5.580242,5.379250,0.686979,0.57000
4,regular3_6|bisection|p1,6,3,0.677008,0.60050,4.031541,4.098251,0.332291,0.28400
5,regular3_6|bisection|p2,6,3,0.846840,0.60550,3.333616,3.734104,0.764491,0.45975
6,regular3_6|maxcut|p1,6,7,1.000000,1.00000,5.939222,5.756000,0.400812,0.33925
7,regular3_6|maxcut|p2,6,7,1.000000,1.00000,6.602949,6.228250,0.720919,0.58450
8,ring_6|bisection|p1,6,2,0.632956,0.53125,2.268502,2.434824,0.575329,0.44025
9,ring_6|bisection|p2,6,2,0.898472,0.69350,2.301063,2.376352,0.804954,0.59700


In [30]:
print(qpu_list.columns.tolist())

['name', 'graph_type', 'n', 'problem', 'p', 'cz_count', 'esp', 'score', 'seed', 'order', 'prediction', 'repeat']


## Repeat on another day
The same 3 circuits as in the main run (p = 1), to measure day-to-day hardware variability.

In [ ]:
raise RuntimeError("DONE 2026-10-27 — this cell creates/submits; do not re-run")

import os
from datetime import date
from qiskit import qpy

repeat = [("ring_6", "maxcut", 1), ("regular3_10", "bisection", 1), ("random_14", "maxcut", 1)]

repeat_calibration = save_calibration_snapshot(backend)
print("calibration:", repeat_calibration)

repeat_labels = []
repeat_circuits = []
for name, problem, p in repeat:
    graph, H, optimum, optimal_bitstrings = problem_setup(name, problem)
    row = find_row(name, problem, p)
    transpiled, seed, order = transpile_qaoa_best(H, p, row["params"], backend)
    repeat_labels.append(name + "|" + problem + "|p" + str(p))
    repeat_circuits.append(transpiled)

sampler = Sampler(mode=backend)
repeat_job = sampler.run(repeat_circuits, shots=4000)

with open("../data/qpu/circuits/" + repeat_job.job_id() + ".qpy", "wb") as f:
    qpy.dump(repeat_circuits, f)
with open("../data/qpu/pending_repeat_" + str(date.today()) + ".json", "w") as f:
    json.dump({"job_id": repeat_job.job_id(), "labels": repeat_labels,
               "calibration": repeat_calibration}, f, indent=2)

print("job id:", repeat_job.job_id())

calibration: ../data/qpu/calibration/ibm_pittsburgh_2026-10-28T10-52.json
job id: dbgt7p4dgr2s73cs88ug


In [ ]:
raise RuntimeError("DONE 2026-10-27 — this cell creates/submits; do not re-run")

from qiskit import qpy

pending_file = "../data/qpu/pending_repeat_2026-10-28.json"   # the file written by cell A
with open(pending_file) as f:
    info = json.load(f)

repeat_job = service.job(info["job_id"])
with open("../data/qpu/circuits/" + info["job_id"] + ".qpy", "rb") as f:
    repeat_circuits = qpy.load(f)

raw_path = save_job_data(repeat_job, info["labels"], repeat_circuits,
                         info["calibration"], tag="repeat")
print("saved:", raw_path, "| QPU seconds:", repeat_job.usage())

saved: ../data/qpu/raw/dbgt7p4dgr2s73cs88ug.json | QPU seconds: 5
